# 🧠 100-Feature Fraud Detection Training Pipeline
**FraudDetectionMLP** — Adaptive architecture trained on real PaySim labels with 100 graph + temporal + topological features.

| Component | Details |
|-----------|--------|
| **Model** | FraudDetectionMLP (3-block MLP, LeakyReLU, He init) |
| **Features** | 100 (PageRank, Centrality, Spectral, TDA, Community, Temporal, Motifs, Signatures, Dataset-specific) |
| **Data** | PaySim 6.3M transactions → 5,500+ graph chunks |
| **Balancing** | SMOTE + WeightedRandomSampler |
| **Target AUC** | > 0.85 |

## 1. Environment Setup & Repository Clone

In [ ]:
import os, sys
from google.colab import drive

# Mount Google Drive for checkpoints & logs
drive.mount('/content/drive')

# Create output directories
DRIVE_BASE = '/content/drive/MyDrive/SOTA_Cluster_Shared'
for d in ['checkpoints', 'hard_samples', 'confusion_matrices', 'metrics']:
    os.makedirs(os.path.join(DRIVE_BASE, d), exist_ok=True)

# Clone repo
REPO = 'multi-domain-project-finance-and-supply-chain-management-'
if not os.path.exists(REPO):
    !git clone https://github.com/nithin12342/{REPO}.git
else:
    %cd {REPO}
    !git pull origin master
    %cd ..

# Add training scripts to path
sys.path.insert(0, os.path.join(REPO, 'backend', 'ai-ml', 'scripts', 'training'))
print('\n✅ Repo cloned & paths configured')

## 2. Install Dependencies

In [ ]:
!pip install -q wandb imbalanced-learn scikit-learn pynvml
print('✅ Dependencies installed')

## 3. Load Data (100 Features)

In [ ]:
import torch
import numpy as np
import pandas as pd
from dataloaders import build_fraud_dataloaders

# Path to the processed features CSV (committed to repo by Codespaces pipeline)
CSV_PATH = os.path.join(REPO, 'backend', 'data', 'processed', 'structural_fraud_features.csv')
assert os.path.exists(CSV_PATH), f'❌ Features CSV not found at {CSV_PATH}. Run the pipeline in Codespaces first!'

# Quick data inspection
df = pd.read_csv(CSV_PATH)
n_features = len([c for c in df.columns if c not in ['chunk_id', 'isFraud']])
fraud_rate = df['isFraud'].mean()
print(f'📊 Dataset: {len(df):,} rows × {n_features} features')
print(f'📊 Fraud rate: {fraud_rate:.4f} ({fraud_rate*100:.2f}%)')
print(f'📊 Fraud: {int(df["isFraud"].sum()):,} | Legit: {int((1-df["isFraud"]).sum()):,}')
del df  # free memory

# Build DataLoaders (SMOTE + StratifiedSplit + StandardScaler)
BATCH_SIZE = 64
train_loader, val_loader, input_dim = build_fraud_dataloaders(
    csv_path=CSV_PATH,
    batch_size=BATCH_SIZE,
    val_split=0.2,
    apply_smote=True
)
print(f'\n✅ DataLoaders ready | input_dim={input_dim}')
print(f'   Train batches: {len(train_loader)} | Val batches: {len(val_loader)}')

## 4. Model Architecture

In [ ]:
from models import FraudDetectionMLP

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'🖥️ Device: {device}')
if device.type == 'cuda':
    print(f'   GPU: {torch.cuda.get_device_name()}')
    print(f'   Memory: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB')

model = FraudDetectionMLP(input_dim=input_dim).to(device)

# Architecture summary
total_params = sum(p.numel() for p in model.parameters())
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'\n🏗️ FraudDetectionMLP(input_dim={input_dim})')
print(f'   Total params:     {total_params:,}')
print(f'   Trainable params: {trainable:,}')
print(model)

## 5. Training Configuration

In [ ]:
import torch.nn as nn
import torch.optim as optim
from sklearn.metrics import roc_auc_score, f1_score, precision_score, recall_score, confusion_matrix

# Hyperparameters
NUM_EPOCHS = 50
LEARNING_RATE = 1e-3
WEIGHT_DECAY = 1e-4
PATIENCE = 8  # Early stopping patience

# Loss: BCEWithLogitsLoss (model outputs raw logits)
criterion = nn.BCEWithLogitsLoss()

# Optimizer: AdamW with weight decay
optimizer = optim.AdamW(
    model.parameters(),
    lr=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY
)

# LR Scheduler: Reduce on plateau
scheduler = optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode='max', factor=0.5, patience=3, verbose=True
)

print(f'⚙️ Config: {NUM_EPOCHS} epochs, LR={LEARNING_RATE}, WD={WEIGHT_DECAY}')
print(f'   Early stopping patience: {PATIENCE}')
print(f'   Scheduler: ReduceLROnPlateau (monitor=val_auc, factor=0.5, patience=3)')

## 6. Training Loop

In [ ]:
from datetime import datetime

best_val_auc = 0.0
epochs_no_improve = 0
history = {'train_loss': [], 'val_loss': [], 'train_auc': [], 'val_auc': [],
           'train_f1': [], 'val_f1': [], 'lr': []}

CKPT_PATH = os.path.join(DRIVE_BASE, 'checkpoints', 'FraudDetectionMLP_best.pt')

def run_epoch(loader, training=True):
    """Run one epoch of training or validation."""
    if training:
        model.train()
    else:
        model.eval()

    total_loss = 0.0
    all_preds = []
    all_labels = []

    ctx = torch.no_grad() if not training else torch.enable_grad()
    with ctx:
        for X_batch, y_batch in loader:
            X_batch = X_batch.to(device)
            y_batch = y_batch.to(device)

            logits = model(X_batch)
            loss = criterion(logits, y_batch)

            if training:
                optimizer.zero_grad()
                loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
                optimizer.step()

            total_loss += loss.item() * len(y_batch)
            probs = torch.sigmoid(logits).cpu().numpy()
            all_preds.extend(probs.tolist())
            all_labels.extend(y_batch.cpu().numpy().tolist())

    avg_loss = total_loss / len(all_labels)
    preds_binary = [1 if p > 0.5 else 0 for p in all_preds]

    try:
        auc = roc_auc_score(all_labels, all_preds)
    except ValueError:
        auc = 0.0

    f1 = f1_score(all_labels, preds_binary, zero_division=0)
    prec = precision_score(all_labels, preds_binary, zero_division=0)
    rec = recall_score(all_labels, preds_binary, zero_division=0)
    acc = sum(1 for a, b in zip(all_labels, preds_binary) if a == b) / len(all_labels)

    return {'loss': avg_loss, 'auc': auc, 'f1': f1, 'precision': prec,
            'recall': rec, 'accuracy': acc, 'labels': all_labels, 'preds': preds_binary}

# === MAIN TRAINING LOOP ===
print('=' * 70)
print('🚀 TRAINING STARTED')
print('=' * 70)

for epoch in range(1, NUM_EPOCHS + 1):
    # Train
    train_metrics = run_epoch(train_loader, training=True)
    # Validate
    val_metrics = run_epoch(val_loader, training=False)

    # LR scheduler step
    current_lr = optimizer.param_groups[0]['lr']
    scheduler.step(val_metrics['auc'])

    # Record history
    history['train_loss'].append(train_metrics['loss'])
    history['val_loss'].append(val_metrics['loss'])
    history['train_auc'].append(train_metrics['auc'])
    history['val_auc'].append(val_metrics['auc'])
    history['train_f1'].append(train_metrics['f1'])
    history['val_f1'].append(val_metrics['f1'])
    history['lr'].append(current_lr)

    # Print epoch summary
    print(f'\n══════ EPOCH {epoch}/{NUM_EPOCHS} ══════')
    print(f'  Train | Loss: {train_metrics["loss"]:.4f} | AUC: {train_metrics["auc"]:.4f} | '
          f'F1: {train_metrics["f1"]:.4f} | Prec: {train_metrics["precision"]:.4f} | '
          f'Rec: {train_metrics["recall"]:.4f}')
    print(f'  Val   | Loss: {val_metrics["loss"]:.4f} | AUC: {val_metrics["auc"]:.4f} | '
          f'F1: {val_metrics["f1"]:.4f} | Prec: {val_metrics["precision"]:.4f} | '
          f'Rec: {val_metrics["recall"]:.4f}')
    print(f'  LR: {current_lr:.2e}')

    # Save best model
    if val_metrics['auc'] > best_val_auc:
        best_val_auc = val_metrics['auc']
        epochs_no_improve = 0
        torch.save({
            'epoch': epoch,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'val_auc': best_val_auc,
            'input_dim': input_dim,
        }, CKPT_PATH)
        print(f'  💾 New best! Saved checkpoint (AUC={best_val_auc:.4f})')
    else:
        epochs_no_improve += 1
        print(f'  ⏳ No improvement ({epochs_no_improve}/{PATIENCE})')

    # Save confusion matrix
    cm = confusion_matrix(val_metrics['labels'], val_metrics['preds'])
    cm_path = os.path.join(DRIVE_BASE, 'confusion_matrices',
                          f'val_cm_epoch_{epoch}.txt')
    with open(cm_path, 'w') as f:
        f.write(f'Epoch {epoch} Validation Confusion Matrix\n')
        f.write(f'TN={cm[0][0]}  FP={cm[0][1]}\n')
        f.write(f'FN={cm[1][0]}  TP={cm[1][1]}\n')
        f.write(f'AUC={val_metrics["auc"]:.4f}  F1={val_metrics["f1"]:.4f}\n')

    # Early stopping
    if epochs_no_improve >= PATIENCE:
        print(f'\n🛑 Early stopping at epoch {epoch} (no improvement for {PATIENCE} epochs)')
        break

print(f'\n{"=" * 70}')
print(f'✅ TRAINING COMPLETE | Best Val AUC: {best_val_auc:.4f}')
print(f'   Checkpoint: {CKPT_PATH}')
print(f'{"=" * 70}')

## 7. Training Curves

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Loss
axes[0].plot(history['train_loss'], label='Train', linewidth=2)
axes[0].plot(history['val_loss'], label='Val', linewidth=2)
axes[0].set_title('Loss', fontsize=14, fontweight='bold')
axes[0].set_xlabel('Epoch')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# AUC
axes[1].plot(history['train_auc'], label='Train', linewidth=2)
axes[1].plot(history['val_auc'], label='Val', linewidth=2)
axes[1].axhline(y=0.85, color='r', linestyle='--', alpha=0.5, label='Target (0.85)')
axes[1].set_title('AUC-ROC', fontsize=14, fontweight='bold')
axes[1].set_xlabel('Epoch')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

# F1
axes[2].plot(history['train_f1'], label='Train', linewidth=2)
axes[2].plot(history['val_f1'], label='Val', linewidth=2)
axes[2].set_title('F1 Score', fontsize=14, fontweight='bold')
axes[2].set_xlabel('Epoch')
axes[2].legend()
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plot_path = os.path.join(DRIVE_BASE, 'metrics', 'training_curves.png')
plt.savefig(plot_path, dpi=150, bbox_inches='tight')
plt.show()
print(f'📊 Saved to {plot_path}')

## 8. Final Evaluation on Best Model

In [ ]:
from sklearn.metrics import classification_report, roc_curve, auc

# Load best checkpoint
ckpt = torch.load(CKPT_PATH, map_location=device)
model.load_state_dict(ckpt['model_state_dict'])
print(f'📦 Loaded best model from epoch {ckpt["epoch"]} (Val AUC={ckpt["val_auc"]:.4f})')

# Final validation
final = run_epoch(val_loader, training=False)

print(f'\n{"═" * 60}')
print(f'📊 FINAL EVALUATION ON BEST MODEL')
print(f'{"═" * 60}')
print(f'  AUC:       {final["auc"]:.4f}')
print(f'  F1:        {final["f1"]:.4f}')
print(f'  Precision: {final["precision"]:.4f}')
print(f'  Recall:    {final["recall"]:.4f}')
print(f'  Accuracy:  {final["accuracy"]:.4f}')
print(f'{"═" * 60}')

# Classification report
print('\n📋 Classification Report:')
print(classification_report(
    final['labels'], final['preds'],
    target_names=['Legitimate', 'Fraud']
))

# Confusion matrix
cm = confusion_matrix(final['labels'], final['preds'])
print(f'Confusion Matrix:')
print(f'  TN={cm[0][0]:5d}  FP={cm[0][1]:5d}')
print(f'  FN={cm[1][0]:5d}  TP={cm[1][1]:5d}')

# ROC Curve
model.eval()
all_probs, all_labels = [], []
with torch.no_grad():
    for X_batch, y_batch in val_loader:
        probs = torch.sigmoid(model(X_batch.to(device))).cpu().numpy()
        all_probs.extend(probs.tolist())
        all_labels.extend(y_batch.numpy().tolist())

fpr, tpr, _ = roc_curve(all_labels, all_probs)
roc_auc_val = auc(fpr, tpr)

plt.figure(figsize=(8, 6))
plt.plot(fpr, tpr, color='darkorange', lw=2, label=f'ROC curve (AUC = {roc_auc_val:.4f})')
plt.plot([0, 1], [0, 1], color='navy', lw=1, linestyle='--')
plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel('False Positive Rate', fontsize=12)
plt.ylabel('True Positive Rate', fontsize=12)
plt.title('ROC Curve — FraudDetectionMLP', fontsize=14, fontweight='bold')
plt.legend(loc='lower right', fontsize=12)
plt.grid(True, alpha=0.3)
roc_path = os.path.join(DRIVE_BASE, 'metrics', 'roc_curve.png')
plt.savefig(roc_path, dpi=150, bbox_inches='tight')
plt.show()
print(f'\n📊 ROC curve saved to {roc_path}')